In [ ]:
# %reset -f

In [ ]:
%pip install matplotlib numpy snntorch

In [ ]:
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import functional as SF
import random

# ==========================================
# 1. ГЕНЕРИРАЊЕ НА ПОДАТОЦИ (Вашиот код)
# ==========================================
def generate_dataset(num_samples, max_n):
    dataset = []
    for _ in range(num_samples):
        # Точен пример (Label 1)
        n = random.randint(1, max_n)
        valid_word = ('a' * n) + ('b' * n)
        dataset.append((valid_word, 1))

        # Неточен пример (Label 0)
        n1 = random.randint(1, max_n)
        n2 = random.randint(1, max_n)
        if n1 == n2:
            n2 += 1 if n2 < max_n else -1
        invalid_word = ('a' * n1) + ('b' * n2)
        dataset.append((invalid_word, 0))

    random.shuffle(dataset)
    return dataset

def word_to_tensor(word):
    """
    Претвора збор во PyTorch тензор со димензии:
    [Time_Steps, Batch_Size(1), Features(2)]
    """
    spike_sequence = []
    for symbol in word:
        if symbol == 'a':
            spike_sequence.append([1.0, 0.0])
        elif symbol == 'b':
            spike_sequence.append([0.0, 1.0])

    # Претворање во тензор [seq_len, 1, 2]
    return torch.tensor(spike_sequence, dtype=torch.float32).unsqueeze(1)

# ==========================================
# 2. КЛАСИЧНА НЕВРОНСКА МРЕЖА (RNN)
# ==========================================
class ClassicRNN(nn.Module):
    def __init__(self, input_size=2, hidden_size=16, num_classes=2):
        super().__init__()
        # Користиме RNN бидејќи работиме со секвенци низ времето
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=False)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x има димензии [time_steps, batch_size, features]
        out, hidden = self.rnn(x)
        # Го земаме само излезот од последниот временски чекор
        last_out = out[-1]
        # Го класифицираме зборот
        return self.fc(last_out)

# ==========================================
# 3. SPIKING NEURAL NETWORK (snnTorch)
# ==========================================
class SpikingNet(nn.Module):
    def __init__(self, input_size=2, hidden_size=16, num_classes=2, beta=0.85):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.lif1 = snn.Leaky(beta=beta)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        self.lif2 = snn.Leaky(beta=beta)

    def forward(self, x):
        time_steps = x.size(0)
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()

        spk2_rec = [] # Ги чуваме импулсите од излезот

        for step in range(time_steps):
            cur_x = x[step]

            cur_x = self.fc1(cur_x)
            spk1, mem1 = self.lif1(cur_x, mem1)

            cur_x = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur_x, mem2)

            spk2_rec.append(spk2)

        return torch.stack(spk2_rec, dim=0)

# ==========================================
# 4. ТРЕНИРАЊЕ И СПОРЕДБА
# ==========================================
def train_and_evaluate():
    print("--- Подготовка на податоци ---")
    # Тренираме на зборови каде n е од 1 до 5
    train_data = generate_dataset(400, max_n=5)
    # Тестираме на ПОДОЛГИ зборови (n од 6 до 10) за да видиме како генерализираат
    test_data = generate_dataset(100, max_n=10)

    rnn_model = ClassicRNN()
    snn_model = SpikingNet()

    # Оптимизатори
    optimizer_rnn = torch.optim.Adam(rnn_model.parameters(), lr=0.01)
    optimizer_snn = torch.optim.Adam(snn_model.parameters(), lr=0.01)

    # Функции за загуба (Loss)
    loss_rnn_fn = nn.CrossEntropyLoss()
    loss_snn_fn = SF.ce_rate_loss() # Специјална загуба за SNN (базирана на броење импулси)

    epochs = 5
    print("\n--- Започнува Тренирањето ---")
    for epoch in range(epochs):
        rnn_loss_total = 0
        snn_loss_total = 0

        # Тренирање
        rnn_model.train()
        snn_model.train()
        for word, label in train_data:
            inputs = word_to_tensor(word)
            targets = torch.tensor([label], dtype=torch.long)

            # RNN Чекор
            optimizer_rnn.zero_grad()
            out_rnn = rnn_model(inputs)
            loss_rnn = loss_rnn_fn(out_rnn, targets)
            loss_rnn.backward()
            optimizer_rnn.step()
            rnn_loss_total += loss_rnn.item()

            # SNN Чекор
            optimizer_snn.zero_grad()
            out_snn = snn_model(inputs)
            loss_snn = loss_snn_fn(out_snn, targets)
            loss_snn.backward()
            optimizer_snn.step()
            snn_loss_total += loss_snn.item()

        print(f"Епоха {epoch+1} | RNN Загуба: {rnn_loss_total/len(train_data):.4f} | SNN Загуба: {snn_loss_total/len(train_data):.4f}")

# ==========================================
    # 5. ТЕСТИРАЊЕ НА ГЕНЕРАЛИЗАЦИЈА
    # ==========================================
    print("\n--- Тестирање на подолги зборови (Генерализација) ---")
    rnn_model.eval()
    snn_model.eval()

    rnn_correct = 0
    snn_correct = 0

    with torch.no_grad():
        for word, label in test_data:
            inputs = word_to_tensor(word)
            targets = label

            # RNN Предвидување
            out_rnn = rnn_model(inputs)
            pred_rnn = out_rnn.argmax(dim=1).item()
            if pred_rnn == targets: rnn_correct += 1

            # SNN Предвидување
            # SF.accuracy_rate враќа директно float (1.0 ако е точно, 0.0 ако е грешно)
            acc_snn = SF.accuracy_rate(out_snn, torch.tensor([targets]))
            if acc_snn == 1.0:
                snn_correct += 1

    print(f"RNN Точност (на долги зборови): {rnn_correct / len(test_data) * 100:.2f}%")
    print(f"SNN Точност (на долги зборови): {snn_correct / len(test_data) * 100:.2f}%")

if __name__ == "__main__":
    train_and_evaluate()